In [1]:
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from tree_search import TreeSearch, Fitter, initialize_thread_pool
from model_fit import DefaultModel, ModelFitter
import tree_search
import model_fit
from prodict import Prodict


In [2]:
import pandas as pd
from fourbynine import fourbynine_board, fourbynine_pattern, fourbynine_move
from parsers import CSVMove

def parse_monkey_4iar_dataframe(df):
    """
    Convert a pandas DataFrame from monkey_4iar format (read_fold_csv) 
    to a list of CSVMove objects for use with model_fit.py
    
    This bypasses CSVMove.create() which has issues with comma-containing participant_ids.
    
    Args:
        df: pandas DataFrame with columns ['black', 'white', 'move', 'color', 'game_id', 'n_pieces']
        default_time: Default reaction time in milliseconds (used if 'rt' column not present)
        group_id: Default group_id to use (can be overridden if needed)
    
    Returns:
        List of CSVMove objects
    """
    # Check required columns
    required_cols = ['black', 'white', 'move', 'color']
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    
    moves = []
    
    for idx, row in df.iterrows():
        # Get board state
        black_pieces = int(row['black'])
        white_pieces = int(row['white'])
        board = fourbynine_board(
            fourbynine_pattern(black_pieces), 
            fourbynine_pattern(white_pieces)
        )
        
        # Convert move bitfield to move index
        move_bitfield = int(row['move'])
        move_index = move_bitfield.bit_length() - 1
        
        # Create fourbynine_move object
        move_obj = fourbynine_move(move_index, 0.0, board.active_player())
        
        # Create CSVMove directly (bypassing CSVMove.create() which has parsing issues)
        csv_move = CSVMove(
            board=board,
            move=move_obj,
            time = 0,
            group_id=1,
            participant_id=1,
            unique_id=row.game_id.replace(",", ".")
        )
        
        moves.append(csv_move)
    
    return moves

In [3]:

data_folder = "../../monkey_4iar/analysis/data/processed/harry/splits_20000"
data = [pd.read_csv(f"{data_folder}/{i}.csv") for i in range(5)]
train_data = data[0][:5]

treesearch = TreeSearch()
treesearch.cutoff = 1.2
fitter = Fitter(treesearch, threads = 1, verbose=True)

defaultmodel = DefaultModel()
defaultmodel.cutoff = 1.2
model_fitter = ModelFitter(args = Prodict({'threads': 1, 'random_sample': False, 'verbose': True}), model = defaultmodel)

In [4]:
assert (treesearch.initial_params == defaultmodel.x0).all()
assert (treesearch.upper_bound == defaultmodel.ub).all()
assert (treesearch.lower_bound == defaultmodel.lb).all()
assert (treesearch.plausible_lower_bound == defaultmodel.plb).all()
assert (treesearch.plausible_upper_bound == defaultmodel.pub).all()

In [5]:
fitter.fit(train_data, manual_seed=1)

Initializing thread pool...
Thread 0: seed=1, Random number: 14089154938208861744


/home/hl4291/ninarow/model_fitting/tree_search.py:347: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data["expected_counts"] = 1


Manual seed: 1
Initial log-likelihood estimation...
Running evaluation with 10 iterations...


100%|██████████| 10/10 [00:00<00:00, 200.89it/s]


bads:TooCloseBounds: For each variable, hard and plausible bounds should not be too close. Moving plausible bounds.
Variables (index) internally transformed to log coordinates: [[0 1]]
	[BADS-0]	 time: 3.09s	 NLL: 1.4757	 Params: [2.001, 0.3, 0.2, 0.1, 1.2, 0.801, 1.001, 0.4, 3.501, 8.0]
Beginning optimization of a STOCHASTIC objective function

 Iteration    f-count      E[f(x)]        SD[f(x)]           MeshScale          Method              Actions
     0           1         1.47571             nan               1                                  


/home/hl4291/ninarow/model_fitting/tree_search.py:352: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data["expected_counts"] = self.calculate_expected_counts(initial_LL, self.model.c).astype(int)


	[BADS-1]	 time: 3.22s	 NLL: 1.0661	 Params: [1.029, 0.194, 0.147, 0.313, 0.539, 0.952, -1.899, -1.372, -4.072, -3.198]
	[BADS-2]	 time: 3.34s	 NLL: 0.78423	 Params: [1.186, 0.029, 0.414, 0.055, 1.939, -4.663, -4.106, 4.424, -3.281, 6.089]
	[BADS-3]	 time: 3.51s	 NLL: 1.0519	 Params: [1.352, 0.321, 0.037, 0.19, 1.082, -3.052, 3.237, 1.782, -0.542, -2.817]
	[BADS-4]	 time: 3.63s	 NLL: 1.0797	 Params: [1.508, 0.048, 0.272, 0.431, 1.346, 1.294, 0.444, -4.99, -2.114, 2.896]
	[BADS-5]	 time: 3.72s	 NLL: 1.74	 Params: [1.776, 0.147, 0.304, 0.16, 1.769, -2.373, 1.753, -1.235, 2.314, 2.368]
	[BADS-6]	 time: 5.01s	 NLL: 1.3977	 Params: [1.933, 0.013, 0.006, 0.348, 0.755, 3.271, 4.585, 3.037, 0.02, 8.022]
	[BADS-7]	 time: 5.03s	 NLL: 0.94361	 Params: [2.089, 0.737, 0.381, 0.451, 1.61, 4.17, -3.071, 0.669, 3.696, -0.884]
	[BADS-8]	 time: 5.12s	 NLL: 1.2548	 Params: [2.245, 0.063, 0.176, 0.253, 0.96, -0.225, -0.903, -2.627, 3.96, 8.462]
	[BADS-9]	 time: 5.26s	 NLL: 1.1982	 Params: [2.321, 0.034, 0

Process ForkPoolWorker-1:
Traceback (most recent call last):
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ^^^^^^^^^^^^^^^^^^^
  File "/home/hl4291/ninarow/model_fitting/tree_search.py", line 233, in parallel_log_likelihood
    predicted_move = self.model.predict(board)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hl4291/ninarow/model_fitting/tree_search.py", line 159, in predict
    search.complete_search()
  File "/home/hl4291/ninarow/model_fitting/fourbynine.py", line 1580, in complete_search
    return _swig_fourbynine.AbstractSearch_complete_search(self)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

Process ForkPoolWorker-2:
Traceback (most recent call last):
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/queues.py", line 365, in get
    res = self._reader.recv_bytes()
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/connection.py", line 216, in recv_bytes
    buf = self._recv_bytes(maxlength)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/connection.py", line 430, in _recv_bytes
    buf = self._recv(4)
          ^^^^^^^^^^^^^
  File "/home/hl4291/.conda/envs/

In [6]:
model_fit.initialize_thread_pool(1, manual_seed=1)
model_fitter.fit_model(parse_monkey_4iar_dataframe(train_data))

Thread 0: Base Seed 1, Seed: 1, Random Number: 14089154938208861744

Setting manual seed 1 for single-thread
Beginning model fit pre-processing: log-likelihood estimation


100%|██████████| 5/5 [00:00<00:00, 36033.54it/s]


bads:TooCloseBounds: For each variable, hard and plausible bounds should not be too close. Moving plausible bounds.
Variables (index) internally transformed to log coordinates: [[0 1]]
	[BADS-0] time: 2.25s	 NLL: 0	 Params: [2.001, 0.3, 0.2, 0.1, 1.2, 0.801, 1.001, 0.4, 3.501, 8.0]
Beginning optimization of a STOCHASTIC objective function

 Iteration    f-count      E[f(x)]        SD[f(x)]           MeshScale          Method              Actions
     0           1               0             nan               1                                  
	[BADS-1] time: 2.26s	 NLL: 0	 Params: [1.029, 0.194, 0.147, 0.313, 0.539, 0.952, -1.899, -1.372, -4.072, -3.198]
	[BADS-2] time: 2.26s	 NLL: 0	 Params: [1.186, 0.029, 0.414, 0.055, 1.939, -4.663, -4.106, 4.424, -3.281, 6.089]
	[BADS-3] time: 2.27s	 NLL: 1	 Params: [1.352, 0.321, 0.037, 0.19, 1.082, -3.052, 3.237, 1.782, -0.542, -2.817]
	[BADS-4] time: 2.27s	 NLL: 1	 Params: [1.508, 0.048, 0.272, 0.431, 1.346, 1.294, 0.444, -4.99, -2.114, 2.896]

Process ForkPoolWorker-3:


	[BADS-109] time: 5.04s	 NLL: 1	 Params: [1.878, 0.297, 0.201, 0.093, 1.202, 0.295, 0.864, 0.227, 3.318, 7.994]


Traceback (most recent call last):
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/queues.py", line 365, in get
    res = self._reader.recv_bytes()
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/connection.py", line 216, in recv_bytes
    buf = self._recv_bytes(maxlength)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multiprocessing/connection.py", line 430, in _recv_bytes
    buf = self._recv(4)
          ^^^^^^^^^^^^^
  File "/home/hl4291/.conda/envs/env/lib/python3.11/multipr

KeyboardInterrupt: 